# Lab 2: Explainable AI สำหรับข้อมูลตาราง (Model-agnostic & SHAP)
**รายวิชา:** จริยธรรมในการใช้ AI และ Explainable AI   
**เวลาโดยประมาณ:** 120 นาที | **Runtime:** CPU เพียงพอ

## วัตถุประสงค์
1. อธิบายแบบจำลองระดับ global ด้วย Permutation Importance, PDP/ICE และ Global Surrogate (พร้อมวัด fidelity)
2. อธิบายระดับ local ด้วย SHAP (TreeSHAP) และ LIME พร้อมตรวจสอบคุณสมบัติ *local accuracy*
3. วัด **เสถียรภาพ (stability)** ของ LIME และความสอดคล้องกับ SHAP
4. สร้าง **counterfactual explanation** ที่คำนึงถึงตัวแปรที่เปลี่ยนได้จริง (actionability)
5. ใช้ SHAP ตรวจสอบบทบาทของตัวแปรอ่อนไหว (sensitive attribute audit)


In [ ]:
!pip -q install shap lime

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import shap
from lime.lime_tabular import LimeTabularExplainer
from xgboost import XGBClassifier
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import roc_auc_score, accuracy_score
from itertools import combinations

RANDOM_STATE = 42

## ส่วนที่ 1 เตรียมข้อมูลและฝึกแบบจำลองกล่องดำ (XGBoost)

In [ ]:
adult = fetch_openml("adult", version=2, as_frame=True)
df = adult.frame.rename(columns={"class": "income"}).copy()
df["y"] = (df["income"].astype(str) == ">50K").astype(int)
X = df.drop(columns=["income", "y", "fnlwgt"])
y = df["y"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

NUM = X.select_dtypes(include="number").columns.tolist()
CAT = [c for c in X.columns if c not in NUM]
prep = ColumnTransformer(
    [("num", "passthrough", NUM),
     ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                       ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                                            min_frequency=50))]), CAT)],
    verbose_feature_names_out=False)

clf = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.9,
                    colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE)
pipe = Pipeline([("prep", prep), ("clf", clf)]).fit(X_tr, y_tr)
print("Test AUC:", round(roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1]), 4))

FEAT = list(pipe.named_steps["prep"].get_feature_names_out())
Z_tr = pd.DataFrame(pipe.named_steps["prep"].transform(X_tr), columns=FEAT, index=X_tr.index)
Z_te = pd.DataFrame(pipe.named_steps["prep"].transform(X_te), columns=FEAT, index=X_te.index)
print("จำนวนตัวแปรหลัง one-hot:", len(FEAT))

## ส่วนที่ 2 การอธิบายระดับ Global
### 2.1 Permutation Importance (บนตัวแปรดิบ ผ่าน pipeline)
$PI_j = s(f; X, y) - \frac{1}{K}\sum_k s(f; X^{\pi_k(j)}, y)$  
**ข้อควรระวัง:** ตัวแปรที่สัมพันธ์กันสูงอาจทำให้ค่า importance ถูกแบ่งกัน หรือสร้างจุดข้อมูลที่ไม่สมจริง

In [ ]:
X_eval = X_te.sample(3000, random_state=RANDOM_STATE); y_eval = y_te.loc[X_eval.index]
pi = permutation_importance(pipe, X_eval, y_eval, scoring="roc_auc", n_repeats=5,
                            random_state=RANDOM_STATE, n_jobs=-1)
pi_df = pd.DataFrame({"mean": pi.importances_mean, "std": pi.importances_std},
                     index=X_eval.columns).sort_values("mean")
pi_df["mean"].plot.barh(xerr=pi_df["std"], figsize=(7, 5), title="Permutation importance (ΔAUC)")
plt.tight_layout(); plt.show()

### 2.2 Partial Dependence (PDP) และ Individual Conditional Expectation (ICE)
เส้นหนา = PDP (ค่าเฉลี่ย), เส้นบาง = ICE ของแต่ละคน หาก ICE ตัดกันมาก แสดงว่ามี interaction ที่ PDP ซ่อนไว้

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
PartialDependenceDisplay.from_estimator(pipe, X_eval, features=["age", "education-num", "hours-per-week"],
                                        kind="both", subsample=200, random_state=RANDOM_STATE,
                                        ax=ax, ice_lines_kw={"alpha": 0.15})
plt.tight_layout(); plt.show()

### 2.3 Global Surrogate และ Fidelity
ฝึกต้นไม้ลึก 3 ชั้นให้เลียนแบบ *คำทำนายของกล่องดำ* (ไม่ใช่ label จริง) แล้ววัด fidelity บนชุดทดสอบ

In [ ]:
bb_tr = pipe.predict(X_tr); bb_te = pipe.predict(X_te)
for depth in [2, 3, 5, 8]:
    sur = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE).fit(Z_tr, bb_tr)
    print(f"depth={depth}: fidelity={accuracy_score(bb_te, sur.predict(Z_te)):.4f} | "
          f"accuracy vs y={accuracy_score(y_te, sur.predict(Z_te)):.4f}")
sur3 = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE).fit(Z_tr, bb_tr)
print(export_text(sur3, feature_names=FEAT))

## ส่วนที่ 3 SHAP (Lundberg & Lee, 2017; Lundberg et al., 2020)
TreeSHAP คำนวณค่า Shapley แบบแม่นตรงสำหรับแบบจำลองต้นไม้ ค่าที่ได้อยู่ในหน่วย **log-odds**

In [ ]:
explainer = shap.TreeExplainer(pipe.named_steps["clf"])
Zs = Z_te.sample(1000, random_state=RANDOM_STATE)
sv = explainer(Zs)

# ตรวจสอบ local accuracy: f(x) = φ0 + Σ φj
margin = pipe.named_steps["clf"].predict(Zs, output_margin=True)
recon = sv.base_values + sv.values.sum(axis=1)
print("Local accuracy (max abs error):", float(np.max(np.abs(margin - recon))))

In [ ]:
shap.plots.beeswarm(sv, max_display=15)

In [ ]:
i = 0
print("คำทำนาย P(>50K) =", round(float(pipe.named_steps['clf'].predict_proba(Zs.iloc[[i]])[0, 1]), 4))
shap.plots.waterfall(sv[i], max_display=12)

In [ ]:
shap.plots.scatter(sv[:, "age"], color=sv)

### 3.1 รวมค่า SHAP ของ one-hot กลับเป็นตัวแปรเดิม (grouped SHAP)
ค่า Shapley มีสมบัติบวกกันได้ (additivity) จึงรวมค่าของคอลัมน์ one-hot ที่มาจากตัวแปรเดียวกันได้

In [ ]:
def group_map(feat_names, num, cat):
    g = {}
    for f in feat_names:
        if f in num: g[f] = f
        else: g[f] = next(c for c in cat if f.startswith(c + "_"))
    return g

gmap = group_map(FEAT, NUM, CAT)
sv_df = pd.DataFrame(sv.values, columns=FEAT, index=Zs.index)
sv_grouped = sv_df.T.groupby(gmap).sum().T
sv_grouped.abs().mean().sort_values().plot.barh(figsize=(7, 5), title="mean |SHAP| (grouped, log-odds)")
plt.tight_layout(); plt.show()

## ส่วนที่ 4 LIME และการทดสอบเสถียรภาพ (Ribeiro et al., 2016)
$\xi(x) = \arg\min_{g \in G} \; \mathcal{L}(f, g, \pi_x) + \Omega(g)$  
LIME สุ่มตัวอย่างรอบ x จึงให้คำอธิบายที่ต่างกันได้เมื่อ seed เปลี่ยน

In [ ]:
bg = Z_tr.sample(5000, random_state=RANDOM_STATE).to_numpy()
predict_fn = lambda z: pipe.named_steps["clf"].predict_proba(pd.DataFrame(z, columns=FEAT))

def lime_explain(x, seed, num_samples=5000, k=8):
    ex = LimeTabularExplainer(bg, feature_names=FEAT, class_names=["<=50K", ">50K"],
                              mode="classification", discretize_continuous=True, random_state=seed)
    return ex.explain_instance(x, predict_fn, num_features=k, num_samples=num_samples)

x0 = Zs.iloc[0].to_numpy()
exp = lime_explain(x0, seed=0)
exp.as_pyplot_figure(); plt.tight_layout(); plt.show()

In [ ]:
def top_features_lime(exp, k=5):
    # แปลงคำอธิบายเงื่อนไข เช่น 'age > 47.00' กลับเป็นชื่อคอลัมน์
    idx = [i for i, _ in exp.as_map()[1]][:k]
    return {FEAT[i] for i in idx}

def jaccard(a, b): return len(a & b) / len(a | b)

rows = []
for ns in [500, 1000, 5000]:
    sets = [top_features_lime(lime_explain(x0, seed=s, num_samples=ns)) for s in range(10)]
    jac = np.mean([jaccard(a, b) for a, b in combinations(sets, 2)])
    rows.append({"num_samples": ns, "mean_pairwise_jaccard_top5": jac})
stab = pd.DataFrame(rows); display(stab)

shap_top5 = set(pd.Series(np.abs(sv.values[0]), index=FEAT).nlargest(5).index)
lime_top5 = top_features_lime(lime_explain(x0, seed=0))
print("SHAP top-5:", shap_top5)
print("LIME top-5:", lime_top5)
print("Jaccard(SHAP, LIME) =", round(jaccard(shap_top5, lime_top5), 3))

## ส่วนที่ 5 Counterfactual Explanation ที่ปฏิบัติได้จริง (Wachter et al., 2018)
$x^{*} = \arg\min_{x'} \; d(x, x') \;\; \text{s.t.} \; f(x') = y'$  
เราค้นหาแบบ grid เฉพาะตัวแปรที่ **เปลี่ยนได้จริง** (education-num เพิ่มได้, hours-per-week ปรับได้ไม่เกิน 60) และตรึงตัวแปรที่เปลี่ยนไม่ได้ เช่น age, sex, race

In [ ]:
def find_counterfactual(x_row, threshold=0.5):
    x_row = x_row.copy()
    ed0, h0 = int(x_row["education-num"]), int(x_row["hours-per-week"])
    grid = [(e, h) for e in range(ed0, max(ed0, 16) + 1) for h in range(h0, max(h0, 60) + 1)]
    cand = pd.concat([x_row.to_frame().T] * len(grid), ignore_index=True).astype(X.dtypes.to_dict())
    cand["education-num"] = [g[0] for g in grid]
    cand["hours-per-week"] = [g[1] for g in grid]
    p = pipe.predict_proba(cand)[:, 1]
    sd_e, sd_h = X_tr["education-num"].std(), X_tr["hours-per-week"].std()
    cost = np.abs(cand["education-num"] - ed0) / sd_e + np.abs(cand["hours-per-week"] - h0) / sd_h
    valid = p >= threshold
    if not valid.any(): return None
    j = np.argmin(np.where(valid, cost, np.inf))
    return {"education-num": (ed0, int(cand.loc[j, "education-num"])),
            "hours-per-week": (h0, int(cand.loc[j, "hours-per-week"])),
            "p_before": float(pipe.predict_proba(x_row.to_frame().T.astype(X.dtypes.to_dict()))[0, 1]),
            "p_after": float(p[j]), "cost": float(cost[j])}

denied = X_te[pipe.predict(X_te) == 0].sample(5, random_state=1)
for idx, row in denied.iterrows():
    print(idx, find_counterfactual(row))

## ส่วนที่ 6 ตรวจสอบบทบาทของตัวแปรอ่อนไหวด้วย SHAP
แบบจำลองนี้ใช้ `sex` และ `race` เป็นตัวแปรอิสระ ลองพิจารณาว่ามีส่วนต่อการทำนายมากน้อยเพียงใด และค่า SHAP ของกลุ่มต่าง ๆ ต่างกันอย่างไร

In [ ]:
sens = sv_grouped[["sex", "race", "relationship", "marital-status"]]
display(sens.abs().mean().to_frame("mean|SHAP|"))
display(sv_grouped.groupby(X_te.loc[Zs.index, "sex"].astype(str))[["sex", "relationship"]].mean())

## แบบฝึกหัดสำหรับรายงาน
1. ตัด `sex` และ `race` ออกแล้วฝึกใหม่ ค่า SHAP ของ `relationship` เปลี่ยนไปอย่างไร เชื่อมโยงกับแนวคิด proxy จาก Lab 1
2. เปรียบเทียบ `shap.TreeExplainer` แบบ `feature_perturbation="interventional"` (ใส่ background data) กับ `"tree_path_dependent"` อธิบายความต่างเชิงทฤษฎี
3. ออกแบบการทดลองวัด stability ของ LIME บน 50 instances และรายงานด้วยสถิติที่เหมาะสม
4. อภิปรายว่า counterfactual ที่ได้ "ปฏิบัติได้จริง" หรือไม่ และเสนอ cost function ที่สะท้อนความยากของการเปลี่ยนแปลงในโลกจริง (algorithmic recourse)
